In [7]:
import os
import shutil
import random
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path
from shapely.geometry import box
import earthaccess as ea
from pathlib import Path
import rioxarray
import contextily as ctx
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import Image, display

In [2]:
root_dir = "SWOT_IW_Dataset"

def get_files_recursive(folder):
    files = set()
    for dirpath, _, filenames in os.walk(folder):
        for f in filenames:
            files.add(os.path.splitext(f)[0])
    return files

# -----------------------------
# Global totals
# -----------------------------
total_data = 0
total_labels = 0
total_matched = 0

for sub in os.listdir(root_dir):
    sub_path = os.path.join(root_dir, sub)
    if not os.path.isdir(sub_path):
        continue

    data_dir = os.path.join(sub_path, "data")
    label_dir = os.path.join(sub_path, "obb_labels")

    if not (os.path.exists(data_dir) and os.path.exists(label_dir)):
        continue

    data_files = get_files_recursive(data_dir)
    label_files = get_files_recursive(label_dir)

    matched = data_files & label_files

    # -----------------------------
    # per-folder stats
    # -----------------------------
    print(f"\n[{sub}]")
    print(f"  data: {len(data_files)} | labels: {len(label_files)} | matched: {len(matched)}")

    # -----------------------------
    # accumulate totals
    # -----------------------------
    total_data += len(data_files)
    total_labels += len(label_files)
    total_matched += len(matched)

# -----------------------------
# final totals
# -----------------------------
print("\n===== TOTAL ACROSS ALL SUBFOLDERS =====")
print(f"Total data files:   {total_data}")
print(f"Total label files:  {total_labels}")
print(f"Total matched:      {total_matched}")


[Andaman Sea]
  data: 694 | labels: 198 | matched: 198

[Northwest Australia]
  data: 751 | labels: 53 | matched: 53

[Celebes Sea]
  data: 657 | labels: 140 | matched: 140

[Indonesia]
  data: 1520 | labels: 280 | matched: 280

[Pacific Central America]
  data: 775 | labels: 38 | matched: 38

[Sulu Sea]
  data: 559 | labels: 143 | matched: 143

[Eastern Equatorial Indian]
  data: 940 | labels: 238 | matched: 238

[Northwest South America]
  data: 798 | labels: 24 | matched: 24

[Western Equatorial Atlantic]
  data: 1414 | labels: 279 | matched: 279

[Eastern Equatorial Pacific]
  data: 3368 | labels: 100 | matched: 100

[Western Equatorial Indian]
  data: 2367 | labels: 217 | matched: 217

[South China Sea two]
  data: 451 | labels: 127 | matched: 127

[South China Sea one]
  data: 771 | labels: 130 | matched: 130

===== TOTAL ACROSS ALL SUBFOLDERS =====
Total data files:   15065
Total label files:  1967
Total matched:      1967


In [5]:
src_root = "SWOT_IW_Dataset"
dst_root = "SWOT_IW_Labeled_Dataset"

def get_files_map(folder):
    """
    returns:
    basename -> full path
    """
    file_map = {}
    for dirpath, _, filenames in os.walk(folder):
        for f in filenames:
            name = os.path.splitext(f)[0]
            file_map[name] = os.path.join(dirpath, f)
    return file_map

os.makedirs(dst_root, exist_ok=True)

for sub in os.listdir(src_root):
    sub_path = os.path.join(src_root, sub)
    if not os.path.isdir(sub_path):
        continue

    data_dir = os.path.join(sub_path, "data")
    label_dir = os.path.join(sub_path, "obb_labels")

    if not (os.path.exists(data_dir) and os.path.exists(label_dir)):
        continue

    data_map = get_files_map(data_dir)
    label_map = get_files_map(label_dir)

    matched_keys = set(data_map.keys()) & set(label_map.keys())

    # destination folders
    dst_data = os.path.join(dst_root, sub, "data")
    dst_label = os.path.join(dst_root, sub, "obb_labels")

    os.makedirs(dst_data, exist_ok=True)
    os.makedirs(dst_label, exist_ok=True)

    for k in matched_keys:
        # copy data file
        shutil.copy2(data_map[k], os.path.join(dst_data, os.path.basename(data_map[k])))

        # copy label file
        shutil.copy2(label_map[k], os.path.join(dst_label, os.path.basename(label_map[k])))

    print(f"[{sub}] Copied {len(matched_keys)} labeled pairs")

[Andaman Sea] Copied 198 labeled pairs
[Northwest Australia] Copied 53 labeled pairs
[Celebes Sea] Copied 140 labeled pairs
[Indonesia] Copied 280 labeled pairs
[Pacific Central America] Copied 38 labeled pairs
[Sulu Sea] Copied 143 labeled pairs
[Eastern Equatorial Indian] Copied 238 labeled pairs
[Northwest South America] Copied 24 labeled pairs
[Western Equatorial Atlantic] Copied 279 labeled pairs
[Eastern Equatorial Pacific] Copied 100 labeled pairs
[Western Equatorial Indian] Copied 217 labeled pairs
[South China Sea two] Copied 127 labeled pairs
[South China Sea one] Copied 130 labeled pairs


In [6]:
# -----------------------------
# Root paths
# -----------------------------
root_dir = "SWOT_IW_Labeled_Dataset"

# -----------------------------
# Convert OBB → YOLO AABB
# -----------------------------
def obb_to_yolo_aabb(coords):
    xmin = coords[:, 0].min()
    xmax = coords[:, 0].max()
    ymin = coords[:, 1].min()
    ymax = coords[:, 1].max()

    xc = (xmin + xmax) / 2
    yc = (ymin + ymax) / 2
    w = xmax - xmin
    h = ymax - ymin

    return xc, yc, w, h

# -----------------------------
# Walk through dataset
# -----------------------------
for dirpath, _, filenames in os.walk(root_dir):

    # only process label folders
    if not dirpath.endswith("obb_labels"):
        continue

    in_dir = dirpath
    out_dir = dirpath.replace("obb_labels", "bb_labels")

    os.makedirs(out_dir, exist_ok=True)

    print(f"\nProcessing folder: {in_dir}")

    for file_name in filenames:
        if not file_name.endswith(".txt"):
            continue

        in_path = os.path.join(in_dir, file_name)
        out_path = os.path.join(out_dir, file_name)

        yolo_lines = []

        with open(in_path, "r") as f:
            for line in f:
                parts = list(map(float, line.strip().split()))
                cls = int(parts[0])
                coords = np.array(parts[1:]).reshape(4, 2)

                xc, yc, w, h = obb_to_yolo_aabb(coords)

                yolo_lines.append(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        with open(out_path, "w") as f:
            f.write("\n".join(yolo_lines))

        print(f"Processed: {file_name}")

print("\nDone. All YOLO bounding boxes saved.")


Processing folder: SWOT_IW_Labeled_Dataset/Andaman Sea/obb_labels
Processed: SWOT_L2_LR_SSH_Expert_034_411_20250623T095527_20250623T104609_PID0_01.txt
Processed: SWOT_L2_LR_SSH_Expert_008_508_20240101T093300_20240101T102347_PIC0_01.txt
Processed: SWOT_L2_LR_SSH_Expert_003_133_20230905T161458_20230905T170626_PGC0_01.txt
Processed: SWOT_L2_LR_SSH_Expert_019_202_20240806T232603_20240807T001731_PIC0_02.txt
Processed: SWOT_L2_LR_SSH_Expert_006_258_20231111T174103_20231111T183232_PGC0_01.txt
Processed: SWOT_L2_LR_SSH_Expert_003_202_20230908T032449_20230908T041617_PGC0_01.txt
Processed: SWOT_L2_LR_SSH_Expert_038_202_20250907T094234_20250907T103403_PID0_01.txt
Processed: SWOT_L2_LR_SSH_Expert_022_508_20241019T120408_20241019T125454_PIC2_01.txt
Processed: SWOT_L2_LR_SSH_Expert_002_230_20230819T064014_20230819T073142_PGC0_01.txt
Processed: SWOT_L2_LR_SSH_Expert_016_230_20240606T091122_20240606T100251_PIC0_01.txt
Processed: SWOT_L2_LR_SSH_Expert_001_202_20230728T095437_20230728T104606_PGC0_01.tx

In [11]:
src_root = "SWOT_IW_Labeled_Dataset"
dst_root = "data"

splits = [5, 10, 20, 50, 100]

random.seed(42)


def get_map(folder):
    """basename -> full path"""
    out = {}
    for f in os.listdir(folder):
        p = os.path.join(folder, f)
        if os.path.isfile(p):
            out[os.path.splitext(f)[0]] = p
    return out


for region in os.listdir(src_root):
    region_path = os.path.join(src_root, region)

    data_dir = os.path.join(region_path, "data")
    obb_dir = os.path.join(region_path, "obb_labels")
    bb_dir = os.path.join(region_path, "bb_labels")

    if not (os.path.exists(data_dir) and os.path.exists(obb_dir) and os.path.exists(bb_dir)):
        continue

    print(f"\nProcessing region: {region}")

    data_map = get_map(data_dir)
    obb_map = get_map(obb_dir)
    bb_map = get_map(bb_dir)

    # only samples that have full labels
    labeled_pool = list(set(data_map) & set(obb_map) & set(bb_map))
    labeled_pool.sort()
    random.shuffle(labeled_pool)

    total = len(labeled_pool)

    for p in splits:
        cutoff = int(total * (p / 100))
        labeled_keys = set(labeled_pool[:cutoff])

        for k in labeled_pool[cutoff:]:
            pass  # (implicitly unlabeled)

        out_base = os.path.join(dst_root, str(p), region)

        labeled_img_dir = os.path.join(out_base, "labeled_data")
        unlabeled_img_dir = os.path.join(out_base, "unlabeled_data")
        obb_out = os.path.join(out_base, "obb_labels")
        bb_out = os.path.join(out_base, "bb_labels")

        os.makedirs(labeled_img_dir, exist_ok=True)
        os.makedirs(unlabeled_img_dir, exist_ok=True)
        os.makedirs(obb_out, exist_ok=True)
        os.makedirs(bb_out, exist_ok=True)

        # -----------------------------
        # labeled + unlabeled split
        # -----------------------------
        for k in data_map.keys():

            src_img = data_map[k]

            if k in labeled_keys:
                # labeled
                shutil.copy2(src_img, os.path.join(labeled_img_dir, os.path.basename(src_img)))
                shutil.copy2(obb_map[k], os.path.join(obb_out, os.path.basename(obb_map[k])))
                shutil.copy2(bb_map[k], os.path.join(bb_out, os.path.basename(bb_map[k])))
            else:
                # unlabeled
                shutil.copy2(src_img, os.path.join(unlabeled_img_dir, os.path.basename(src_img)))

        print(f"[{region}] {p}% -> labeled: {len(labeled_keys)} | unlabeled: {len(data_map) - len(labeled_keys)}")

print("\nDone.")


Processing region: Andaman Sea
[Andaman Sea] 5% -> labeled: 9 | unlabeled: 189
[Andaman Sea] 10% -> labeled: 19 | unlabeled: 179
[Andaman Sea] 20% -> labeled: 39 | unlabeled: 159
[Andaman Sea] 50% -> labeled: 99 | unlabeled: 99
[Andaman Sea] 100% -> labeled: 198 | unlabeled: 0

Processing region: Northwest Australia
[Northwest Australia] 5% -> labeled: 2 | unlabeled: 51
[Northwest Australia] 10% -> labeled: 5 | unlabeled: 48
[Northwest Australia] 20% -> labeled: 10 | unlabeled: 43
[Northwest Australia] 50% -> labeled: 26 | unlabeled: 27
[Northwest Australia] 100% -> labeled: 53 | unlabeled: 0

Processing region: Celebes Sea
[Celebes Sea] 5% -> labeled: 7 | unlabeled: 133
[Celebes Sea] 10% -> labeled: 14 | unlabeled: 126
[Celebes Sea] 20% -> labeled: 28 | unlabeled: 112
[Celebes Sea] 50% -> labeled: 70 | unlabeled: 70
[Celebes Sea] 100% -> labeled: 140 | unlabeled: 0

Processing region: Indonesia
[Indonesia] 5% -> labeled: 14 | unlabeled: 266
[Indonesia] 10% -> labeled: 28 | unlabeled: